# 02 · Coleta de dados

**Objetivo:** coletar os resultados de corrida e classificação de todas as etapas desde 2018, reaproveitando o processo construído no notebook 01, e salvar tudo numa única tabela.

## 1. Setup


In [30]:
import os 
import fastf1
import pandas as pd
fastf1.set_log_level("ERROR")

In [31]:
fastf1.Cache.enable_cache('../cache')  

## 2. Função de coleta de uma corrida
Reaproveito o processo do notebook 01 numa função que recebe o ano e o nome do GP. Ela carrega a corrida e a classificação, junta as duas pelo `DriverId` (LEFT JOIN) e adiciona as colunas `Ano` e `GP`, para identificar de qual corrida veio cada linha.

In [ ]:
# Carrega corrida e classificação de um GP e junta numa tabela (mesmo processo do notebook 01)
def carregar_corrida(ano, gp):
    corrida = fastf1.get_session(ano, gp, "R")
    corrida.load(laps=False, telemetry=False, weather=False, messages=False)
    resultado_corrida = corrida.results

    quali = fastf1.get_session(ano, gp, "Q")
    quali.load(laps=False, telemetry=False, weather=False, messages=False)
    resultado_quali = quali.results

    quali_enxuta = resultado_quali[["DriverId", "Position", "Q1", "Q2", "Q3"]].rename(
        columns={"Position": "QualiPosition"}
    )
    corrida_sem_quali = resultado_corrida.drop(columns=["Q1", "Q2", "Q3"])
    base = pd.merge(corrida_sem_quali, quali_enxuta, on="DriverId", how="left")
    base["Ano"] = ano
    base["GP"] = gp
    return base

## 3. Calendário
Para coletar todas as corridas, preciso da lista de GPs de cada temporada. O FastF1 fornece o calendário oficial com `get_event_schedule`, que traz o número da etapa, o nome do GP, o país e a data. Os testes de pré-temporada são excluídos, porque não são corridas.

In [ ]:
# Calendário da temporada 2024, sem os testes de pré-temporada
calendario = fastf1.get_event_schedule(2024, include_testing=False)
calendario[["RoundNumber", "EventName", "Country", "EventDate"]]

### Teste: coletar uma temporada completa (2024)
Antes de coletar todos os anos, testo o processo com uma única temporada. Escolhi **2024** porque é uma temporada **completa** (todas as 24 corridas já aconteceram), diferente de 2026, que ainda está em andamento. Se funcionar com um ano, repito para os demais.

In [ ]:
# Percorre os 24 GPs de 2024, carrega cada um e empilha numa tabela só
tabelas = []
for gp in calendario["EventName"]:
    tabela = carregar_corrida(2024, gp)
    tabelas.append(tabela)

temporada = pd.concat(tabelas, ignore_index=True)   # empilha tudo numa tabela só
temporada.shape

**Resultado:** a temporada 2024 tem 479 linhas. A Austrália teve só 19 carros (a Williams correu com um carro só), por isso não deu 480.

## 4. Função da temporada
Transformo o loop acima numa função que recebe o ano. Assim, consigo coletar qualquer temporada com uma única linha.

In [ ]:
# Carrega todas as corridas já disputadas de uma temporada e empilha numa tabela só
def carregar_temporada(ano):
    calendario = fastf1.get_event_schedule(ano, include_testing=False)
    hoje = pd.Timestamp.today()
    calendario = calendario[calendario["EventDate"] < hoje]  # só pega corridas já realizadas
    tabelas = []

    for gp in calendario["EventName"]:
        tabela = carregar_corrida(ano, gp)
        tabelas.append(tabela)
    return pd.concat(tabelas, ignore_index=True)   # empilha tudo numa tabela só

### Teste: temporada em andamento (2026)
Testo a função com 2026 para confirmar que o filtro de data funciona: só devem entrar as corridas que já aconteceram.

In [ ]:
dados_2026 = carregar_temporada(2026)
dados_2026.shape

In [ ]:
dados_2026["GP"].value_counts()

## 5. Coleta de todas as temporadas
Rodo a função para cada ano de 2018 a 2026 e salvo cada temporada num arquivo CSV na pasta `data`, para não precisar baixar de novo.

In [35]:
for ano in range(2018, 2027):
    arquivo = f"../data/dados_{ano}.csv"

    if os.path.exists(arquivo):
        print(ano, "já salvo, pulando")
        continue

    dados = carregar_temporada(ano)
    dados.to_csv(arquivo, index=False)
    print(ano, dados.shape)

2018 já salvo, pulando
2019 já salvo, pulando
2020 já salvo, pulando
2021 já salvo, pulando


Request for URL https://raw.githubusercontent.com/theOehrly/f1schedule/master/schedule_2022.json failed; using cached response
Traceback (most recent call last):
  File "c:\Users\yuri.silva\ML-study\Formula1\.venv\Lib\site-packages\requests_cache\session.py", line 311, in _resend
    response = self._send_and_cache(request, actions, cached_response, **kwargs)
  File "c:\Users\yuri.silva\ML-study\Formula1\.venv\Lib\site-packages\requests_cache\session.py", line 279, in _send_and_cache
    response = super().send(request, **kwargs)
  File "c:\Users\yuri.silva\ML-study\Formula1\.venv\Lib\site-packages\fastf1\req.py", line 110, in send
    lim.limit()
    ~~~~~~~~~^^
  File "c:\Users\yuri.silva\ML-study\Formula1\.venv\Lib\site-packages\fastf1\req.py", line 80, in limit
    raise RateLimitExceededError(self._info)
fastf1.exceptions.RateLimitExceededError: any API: 500 calls/h
Request for URL https://raw.githubusercontent.com/theOehrly/f1schedule/master/schedule_2022.json failed; using cache

RateLimitExceededError: any API: 500 calls/h